# AIC 2026 — Caption toàn bộ keyframe bằng Florence-2-large-ft

Chạy được cả **Colab** và **Kaggle 2×T4**. Việc duy nhất: sinh caption cho mọi ảnh trong `TARGET_FOLDERS`, mỗi video ghi một file `<video_id>.json`.

Kaggle: bật **Settings → Accelerator → GPU T4 ×2** và **Internet → On**.

Chạy lần lượt 7 cell từ trên xuống. Session bị ngắt thì chạy lại từ đầu — video nào xong rồi sẽ được bỏ qua.

In [ ]:
!nvidia-smi

# KHÔNG đụng vào torch/numpy/Pillow của môi trường. Florence-2 cần thêm timm + einops.
!pip -q install -U transformers accelerate
!pip -q install -q timm einops

In [ ]:
import os
import sys
from pathlib import Path

IS_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or Path('/kaggle/input').exists()
IS_COLAB = (not IS_KAGGLE) and ('google.colab' in sys.modules or Path('/content').is_dir())

if IS_KAGGLE:
    DATASET_DIRECTORY = Path('/kaggle/input/datasets/fatle542/aic-dataset')
    CAPTION_DIRECTORY = Path('/kaggle/working/ImageCaptioning_Florence-2-large-ft')
    SCRATCH_DIRECTORY = Path('/kaggle/working/_scratch')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Dataset_Directory')
    CAPTION_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/ImageCaptioning_Florence-2-large-ft')
    SCRATCH_DIRECTORY = Path('/content')

TARGET_FOLDERS = [
    'Keyframes_L21', 'Keyframes_L22', 'Keyframes_L23', 'Keyframes_L24',
    'Keyframes_L25', 'Keyframes_L26_a', 'Keyframes_L26_b',
    'Keyframes_L26_c', 'Keyframes_L26_d', 'Keyframes_L26_e',
    'Keyframes_L27', 'Keyframes_L28', 'Keyframes_L29', 'Keyframes_L30',
]

# --- Model ------------------------------------------------------------------
# Cùng trọng số, khác ĐỊNH DẠNG: bản florence-community chạy thẳng bằng class có
# sẵn trong transformers; bản microsoft cần remote code, chỉ dùng khi bản kia hỏng.
NATIVE_MODEL_ID = 'florence-community/Florence-2-large-ft'
REMOTE_MODEL_ID = 'microsoft/Florence-2-large-ft'
MODEL_ID = REMOTE_MODEL_ID     # nhãn ghi vào JSON, giữ cố định

# Florence-2 KHÔNG nhận prompt tự do, chỉ hiểu task token cố định:
#   '<CAPTION>' ngắn | '<DETAILED_CAPTION>' 1-2 câu | '<MORE_DETAILED_CAPTION>' 3-6 câu
TASK = '<MORE_DETAILED_CAPTION>'
MAX_NEW_TOKENS = 256
NUM_BEAMS = 3            # 1 = nhanh gấp đôi, caption khô hơn
BATCH_SIZE = 8           # số ảnh mỗi lượt gọi model, TÍNH TRÊN MỖI GPU; giảm nếu OOM

# Keyframe liên tiếp trong một shot gần như giống hệt nhau -> gom bằng dHash, chỉ
# ảnh đại diện mới tốn GPU, ảnh trùng chép lại caption. JSON vẫn đủ mọi keyframe.
#   0 = chỉ gộp ảnh gần y hệt | 4 = mặc định | -1 = tắt, caption mọi ảnh
DEDUP_MAX_DISTANCE = 4
DEDUP_HASH_SIZE = 8

# Dừng giữa hai video sau ngần này giờ, để kịp đóng gói trước khi Kaggle cắt ở 12h.
# 0 = chạy tới khi hết việc hoặc bị cắt.
MAX_RUN_HOURS = 10.5 if IS_KAGGLE else 0

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

# --- Dò đường dẫn thật ------------------------------------------------------
# Dataset gắn trên Kaggle hay lồng thêm một cấp (tên dataset lặp lại) -> tự tụt xuống.
def _looks_like_dataset_root(path):
    if not path.is_dir():
        return False
    return any(p.is_dir() and (p.name.startswith('Keyframes') or p.name.startswith('map-keyframes'))
               for p in path.iterdir())

if not _looks_like_dataset_root(DATASET_DIRECTORY) and DATASET_DIRECTORY.is_dir():
    for _child in sorted(p for p in DATASET_DIRECTORY.iterdir() if p.is_dir()):
        if _looks_like_dataset_root(_child):
            print(f'Dataset thật nằm sâu hơn một cấp -> {_child}')
            DATASET_DIRECTORY = _child
            break

def _find_map_keyframes(root):
    direct = root / 'map-keyframes-aic25-b1' / 'map-keyframes'
    if direct.is_dir():
        return direct
    for pattern in ('map-keyframes', '*/map-keyframes', 'map-keyframes*/map-keyframes*'):
        for candidate in sorted(root.glob(pattern)):
            if candidate.is_dir():
                return candidate
    return direct

dataset_root = DATASET_DIRECTORY.resolve()
if not dataset_root.is_dir() and IS_KAGGLE:
    print('KHÔNG thấy', dataset_root, '- đang có trong /kaggle/input:')
    for entry in sorted(Path('/kaggle/input').glob('*')):
        print(' ', entry)
        for sub in sorted(entry.glob('*'))[:10]:
            print('    ', sub.name)
assert dataset_root.is_dir(), f'Không thấy DATASET_DIRECTORY: {dataset_root}'

MAP_KEYFRAMES_DIRECTORY = _find_map_keyframes(dataset_root)
assert MAP_KEYFRAMES_DIRECTORY.is_dir(), f'Thiếu map-keyframes: {MAP_KEYFRAMES_DIRECTORY}'

KEYFRAME_ROOTS, missing = [], []
for folder in TARGET_FOLDERS:
    selected = (dataset_root / folder.strip()).resolve()
    assert dataset_root in selected.parents, f'Chỉ nhận đường dẫn tương đối: {folder}'
    if selected.is_dir(): KEYFRAME_ROOTS.append(selected)
    else: missing.append(folder)
assert KEYFRAME_ROOTS, ('Không thấy thư mục keyframe nào. Trong dataset có: '
                        + ', '.join(sorted(p.name for p in dataset_root.iterdir() if p.is_dir())[:20]))

OUTPUT_ROOT = CAPTION_DIRECTORY.resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = SCRATCH_DIRECTORY / 'caption_checkpoint'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = f'{MODEL_ID}|{TASK}'

if missing:
    print('Bỏ qua thư mục không tồn tại:', ', '.join(missing))
print('Dataset:', dataset_root)
print('Map    :', MAP_KEYFRAMES_DIRECTORY)
print('Output :', OUTPUT_ROOT)
print(f'Sẽ caption {len(KEYFRAME_ROOTS)}/{len(TARGET_FOLDERS)} thư mục keyframe')

In [ ]:
import csv
import re

VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

def find_video_dirs(root):
    base = root / 'keyframes'
    if not base.is_dir(): base = root
    return [p for p in sorted(base.iterdir()) if p.is_dir() and VIDEO_ID_PATTERN.match(p.name)]

def load_keyframe_map(video_id):
    """frame_idx / pts_time để nối caption về timeline của video."""
    path = MAP_KEYFRAMES_DIRECTORY / f'{video_id}.csv'
    if not path.is_file(): return None
    mapping = {}
    with path.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            try:
                mapping[int(row['n'])] = {'pts_time': float(row['pts_time']),
                                          'fps': float(row['fps']),
                                          'frame_idx': int(row['frame_idx'])}
            except (KeyError, TypeError, ValueError): pass
    return mapping or None

def keyframe_order(path):
    return int(path.stem) if path.stem.isdigit() else None

def list_images(video_dir):
    return sorted((p for p in video_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
                  key=lambda p: (keyframe_order(p) is None, keyframe_order(p) or 0, p.name))

video_dirs = sorted((v for root in KEYFRAME_ROOTS for v in find_video_dirs(root)), key=lambda p: p.name)
total_images = sum(len(list_images(v)) for v in video_dirs)
have_map = sum(load_keyframe_map(v.name) is not None for v in video_dirs)
print(f'{len(video_dirs)} video | {total_images} keyframe | {have_map} video có map-keyframes')

In [ ]:
import torch
from transformers import AutoProcessor

device_count = torch.cuda.device_count()
if device_count:
    for gid in range(device_count):
        prop = torch.cuda.get_device_properties(gid)
        print(f'GPU {gid}: {prop.name}, {prop.total_memory / 1024 ** 3:.0f} GB')
else:
    print('CẢNH BÁO: không có GPU, chạy CPU sẽ cực chậm.')

# T4 (Turing) không có bf16 -> fp16 là lựa chọn đúng và duy nhất hợp lý trên GPU.
torch_dtype = torch.float16 if device_count else torch.float32

def _from_pretrained(cls, model_id, **extra):
    """transformers >= 4.56 đổi tên torch_dtype -> dtype; thử bản mới trước."""
    try:
        return cls.from_pretrained(model_id, dtype=torch_dtype, **extra)
    except TypeError:
        return cls.from_pretrained(model_id, torch_dtype=torch_dtype, **extra)

def load_native():
    """Checkpoint đã convert + class có sẵn. KHÔNG truyền trust_remote_code:
    cờ đó ép quay lại code của repo và sẽ vỡ."""
    from transformers import AutoModelForImageTextToText
    return (AutoProcessor.from_pretrained(NATIVE_MODEL_ID),
            _from_pretrained(AutoModelForImageTextToText, NATIVE_MODEL_ID))

def load_remote_code():
    """Dự phòng cho transformers cũ: repo microsoft/, vá bỏ dòng import flash_attn
    mà code khai báo nhưng không thực sự dùng (cài flash_attn thật mất 10-30 phút build)."""
    from unittest.mock import patch
    from transformers import AutoModelForCausalLM
    from transformers.dynamic_module_utils import get_imports

    def imports_without_flash_attn(filename):
        return [name for name in get_imports(filename) if name != 'flash_attn']

    with patch('transformers.dynamic_module_utils.get_imports', imports_without_flash_attn):
        return (AutoProcessor.from_pretrained(REMOTE_MODEL_ID, trust_remote_code=True),
                _from_pretrained(AutoModelForCausalLM, REMOTE_MODEL_ID, trust_remote_code=True))

_WORKING_LOADER = None

def load_one(device):
    """MỘT bản model đầy đủ + processor RIÊNG, đặt trọn trên MỘT device.

    Không dùng device_map='auto': cờ đó chia layer ra nhiều GPU và chúng chạy nối
    đuôi nhau, không nhanh hơn. Florence-2 fp16 chỉ ~1.6GB nên mỗi T4 chứa trọn
    một bản, hai GPU chạy hai batch khác nhau mới là thứ cho gần x2 tốc độ.
    Processor riêng cho từng worker để hai thread không dùng chung state nào.
    """
    global _WORKING_LOADER
    if _WORKING_LOADER == 'remote':
        processor, model = load_remote_code()
    else:
        try:
            processor, model = load_native()
            _WORKING_LOADER = 'native'
        except Exception as exc:
            print(f'  Bản native không dùng được ({type(exc).__name__}: {exc}) -> lùi về remote code')
            processor, model = load_remote_code()
            _WORKING_LOADER = 'remote'
    model.to(device).eval()
    # Encoder-decoder (DaViT + BART): prompt vào ENCODER nên padding phải là đúng.
    getattr(processor, 'tokenizer', processor).padding_side = 'right'
    return processor, model


class Worker:
    def __init__(self, device, processor, model):
        self.device = torch.device(device)
        self.processor = processor
        self.model = model


WORKERS = []
for _device in ([f'cuda:{i}' for i in range(device_count)] or ['cpu']):
    print('Đang tải model lên', _device, '...')
    WORKERS.append(Worker(_device, *load_one(_device)))

print(f'Xong: {len(WORKERS)} worker ({_WORKING_LOADER}) | mỗi lượt xử lý '
      f'{BATCH_SIZE * len(WORKERS)} ảnh | task {TASK}')

In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageOps

# Florence-2 hay mở đầu "The image shows..." — không mang thông tin nhưng chiếm chỗ
# trong embedding và làm mọi caption trông giống nhau -> cắt bỏ.
META_PREFIXES = (
    'the image shows', 'the image depicts', 'the image features', 'the image captures',
    'this image shows', 'this image depicts', 'the photo shows', 'the picture shows',
    'the video keyframe shows', 'this keyframe shows', 'the keyframe shows',
    'the scene shows', 'the frame shows', 'in this image,', 'in this keyframe,',
    'in the image,', 'here we see', 'we see', 'it shows',
)
SENTENCE_SPLIT = re.compile(r'(?<=[.!?])\s+')

def clean_caption(text):
    text = ' '.join((text or '').split())
    lowered = text.lower()
    for prefix in META_PREFIXES:
        if lowered.startswith(prefix):
            text = text[len(prefix):].lstrip(' ,:')
            text = text[:1].upper() + text[1:]
            break
    # Bỏ câu cuối bị cắt dở khi chạm trần token.
    parts = SENTENCE_SPLIT.split(text)
    if len(parts) > 1 and not parts[-1].rstrip().endswith(('.', '!', '?')):
        parts = parts[:-1]
    return ' '.join(parts).strip()

def load_image(image_path):
    return ImageOps.exif_transpose(Image.open(image_path)).convert('RGB')

@torch.inference_mode()
def caption_batch(worker, paths):
    """Caption một batch ảnh trên MỘT GPU, trả về list[str] cùng thứ tự với paths."""
    images = [load_image(p) for p in paths]
    model, proc = worker.model, worker.processor
    inputs = proc(text=[TASK] * len(images), images=images, return_tensors='pt', padding=True)
    generated = model.generate(
        input_ids=inputs['input_ids'].to(worker.device),
        # pixel_values phải đúng dtype của model, nếu không fp16 báo lỗi kiểu.
        pixel_values=inputs['pixel_values'].to(worker.device, dtype=model.dtype),
        max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS, do_sample=False,
        early_stopping=NUM_BEAMS > 1,
    )
    # skip_special_tokens=False: post_process_generation cần token điều khiển còn nguyên.
    decoded = proc.batch_decode(generated, skip_special_tokens=False)
    return [clean_caption(proc.post_process_generation(text, task=TASK, image_size=image.size)[TASK])
            for text, image in zip(decoded, images)]

def caption_batch_safe(worker, paths):
    try:
        return caption_batch(worker, paths)
    except torch.cuda.OutOfMemoryError:
        # Rơi về từng ảnh một thay vì hỏng cả video.
        with torch.cuda.device(worker.device):
            torch.cuda.empty_cache()
        print(f'      OOM trên {worker.device} -> chạy từng ảnh (cân nhắc giảm BATCH_SIZE)')
        return [caption_batch(worker, [path])[0] for path in paths]

EXECUTOR = ThreadPoolExecutor(max_workers=max(1, len(WORKERS)))

def caption_paths(image_paths):
    """Chia batch cho các GPU chạy song song, GIỮ NGUYÊN thứ tự vào/ra.

    Duyệt future theo thứ tự submit chứ KHÔNG dùng as_completed — thứ tự này là
    thứ giữ cho caption khớp đúng keyframe.
    """
    batches = [image_paths[i:i + BATCH_SIZE] for i in range(0, len(image_paths), BATCH_SIZE)]
    if not batches:
        return []
    if len(WORKERS) == 1 or len(batches) == 1:
        return [row for batch in batches for row in caption_batch_safe(WORKERS[0], batch)]
    futures = [EXECUTOR.submit(caption_batch_safe, WORKERS[i % len(WORKERS)], batch)
               for i, batch in enumerate(batches)]
    return [row for future in futures for row in future.result()]

def frame_hash(image_path):
    """dHash 64-bit. draft() cho JPEG decode ở độ phân giải thấp -> nhanh hơn nhiều lần."""
    with Image.open(image_path) as image:
        image.draft('L', (DEDUP_HASH_SIZE * 4, DEDUP_HASH_SIZE * 4))
        small = image.convert('L').resize((DEDUP_HASH_SIZE + 1, DEDUP_HASH_SIZE),
                                          Image.Resampling.BILINEAR)
    pixels = list(small.getdata())
    bits = 0
    for row in range(DEDUP_HASH_SIZE):
        base = row * (DEDUP_HASH_SIZE + 1)
        for col in range(DEDUP_HASH_SIZE):
            bits = (bits << 1) | int(pixels[base + col] > pixels[base + col + 1])
    return bits

def group_duplicates(image_paths):
    """[(ảnh đại diện, [ảnh trùng...]), ...] — chỉ ảnh đại diện mới đưa vào model."""
    if DEDUP_MAX_DISTANCE < 0:
        return [(path, []) for path in image_paths]
    groups, reference = [], None
    for path in image_paths:
        digest = frame_hash(path)
        # So với ảnh ĐẠI DIỆN chứ không phải ảnh liền trước: tránh cảnh pan chậm
        # trôi dần từng chút rồi gộp nhầm cả đoạn dài thành một nhóm.
        if groups and bin(digest ^ reference).count('1') <= DEDUP_MAX_DISTANCE:
            groups[-1][1].append(path)
        else:
            groups.append((path, []))
            reference = digest
    return groups

In [ ]:
import json
import shutil

RESULT_GLOB = 'L[0-9][0-9]_V[0-9][0-9][0-9].json'
WORK_CHUNK = BATCH_SIZE * max(1, len(WORKERS))   # đủ việc cho mọi GPU mỗi vòng

def output_json_path(video_id): return OUTPUT_ROOT / f'{video_id}.json'
def partial_path(video_id): return CHECKPOINT_DIR / f'{video_id}.partial.json'

def atomic_write(path, payload):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)

def is_done(video_id):
    path = output_json_path(video_id)
    if not path.is_file(): return False
    try: payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception: return False
    return payload.get('run_id') == RUN_ID and bool(payload.get('complete'))

def restore_previous_results():
    """/kaggle/working bị xoá khi session kết thúc, nên đợt sau notebook sẽ không
    thấy gì và caption lại từ đầu. Gắn output đợt trước làm input (Add Input ->
    Your Work -> Notebook Output) rồi cell này chép ngược về thư mục output."""
    if not IS_KAGGLE or not Path('/kaggle/input').is_dir():
        return 0
    copied = 0
    for pattern in ('*', '*/*', '*/*/*'):
        for candidate in sorted(Path('/kaggle/input').glob(pattern)):
            # Bỏ qua nhánh dataset keyframe: hàng trăm nghìn ảnh, quét vào chỉ tốn thời gian.
            if not candidate.is_dir() or candidate == dataset_root or dataset_root in candidate.parents:
                continue
            if candidate.resolve() == OUTPUT_ROOT:
                continue
            for source in sorted(candidate.glob(RESULT_GLOB)):
                target = OUTPUT_ROOT / source.name
                # Bản đang ở output có thể mới hơn -> chỉ ghi đè khi bản cũ dài hơn.
                if target.exists() and target.stat().st_size >= source.stat().st_size:
                    continue
                shutil.copy2(source, target)
                copied += 1
    return copied

def pack_outputs():
    """Nén output thành một zip để tải một phát ở panel Output."""
    if not IS_KAGGLE:
        return
    archive = shutil.make_archive('/kaggle/working/florence2_captions', 'zip', OUTPUT_ROOT)
    print(f'Đã nén: {archive} ({Path(archive).stat().st_size / 1024 ** 2:.1f} MB)')

def caption_video(video_dir, progress_every=200):
    video_id = video_dir.name
    mapping = load_keyframe_map(video_id)
    images = list_images(video_dir)

    payload = None
    if partial_path(video_id).is_file():
        try:
            saved = json.loads(partial_path(video_id).read_text(encoding='utf-8'))
            if saved.get('run_id') == RUN_ID:
                payload = saved
        except Exception:
            payload = None
    if payload is None:
        payload = {'video_id': video_id, 'source': str(video_dir), 'model': MODEL_ID,
                   'run_id': RUN_ID, 'task': TASK, 'language': 'en',
                   'num_beams': NUM_BEAMS, 'max_new_tokens': MAX_NEW_TOKENS,
                   'dedup_max_distance': DEDUP_MAX_DISTANCE,
                   'has_keyframe_map': mapping is not None, 'complete': False,
                   'keyframe_count': 0, 'captioned_count': 0, 'keyframes': []}

    done = {item['keyframe'] for item in payload['keyframes']}
    pending = [p for p in images if p.name not in done]
    groups = group_duplicates(pending)
    if pending and len(groups) < len(pending):
        print(f'    {len(pending)} keyframe -> {len(groups)} ảnh cần model '
              f'({100 * (1 - len(groups) / len(pending)):.0f}% trùng)')

    def record(image_path, caption, duplicate_of):
        order = keyframe_order(image_path)
        mapped = mapping.get(order) if mapping and order is not None else None
        payload['keyframes'].append({
            'keyframe': image_path.name, 'n': order,
            'frame_idx': mapped['frame_idx'] if mapped else None,
            'pts_time': mapped['pts_time'] if mapped else None,
            'fps': mapped['fps'] if mapped else None,
            'caption': caption, 'duplicate_of': duplicate_of,
        })

    for start in range(0, len(groups), WORK_CHUNK):
        chunk = groups[start:start + WORK_CHUNK]
        for (representative, duplicates), caption in zip(chunk, caption_paths([g[0] for g in chunk])):
            record(representative, caption, None)
            for duplicate in duplicates:
                record(duplicate, caption, representative.name)   # ảnh trùng chép lại caption
        payload['keyframe_count'] = len(payload['keyframes'])
        payload['captioned_count'] = sum(k['duplicate_of'] is None for k in payload['keyframes'])
        atomic_write(partial_path(video_id), payload)
        processed = min(start + WORK_CHUNK, len(groups))
        if progress_every and processed % progress_every < WORK_CHUNK:
            print(f'    {processed}/{len(groups)} ảnh')

    payload['complete'] = True
    payload['keyframes'].sort(key=lambda k: (k['n'] is None, k['n'] or 0, k['keyframe']))
    atomic_write(output_json_path(video_id), payload)
    if partial_path(video_id).exists():
        partial_path(video_id).unlink()
    return payload

In [ ]:
import time
import traceback

restored = restore_previous_results()
if restored:
    print(f'Nạp lại {restored} file JSON từ các đợt trước.')

pending_videos = [v for v in video_dirs if not is_done(v.name)]
print(f'{len(video_dirs)} video: đã xong {len(video_dirs) - len(pending_videos)}, '
      f'còn {len(pending_videos)}\n')

success = failed = 0
failures = []
started_all = time.time()
# Dừng giữa HAI video chứ không để Kaggle chém giữa chừng, để còn kịp đóng gói.
deadline = started_all + MAX_RUN_HOURS * 3600 if MAX_RUN_HOURS else None
stopped_early = False

for index, video_dir in enumerate(pending_videos, 1):
    if deadline and time.time() > deadline:
        stopped_early = True
        print(f'\nĐã chạy đủ {MAX_RUN_HOURS}h -> dừng, còn {len(pending_videos) - index + 1} video.')
        break
    print(f'[{index}/{len(pending_videos)}] {video_dir.name}')
    started = time.time()
    try:
        payload = caption_video(video_dir)
        print(f"    {payload['keyframe_count']} keyframe "
              f"({payload['captioned_count']} lượt gọi model, {time.time() - started:.0f}s)")
        success += 1
    except Exception as exc:
        failed += 1
        failures.append({'video_id': video_dir.name, 'error': repr(exc)})
        traceback.print_exc()
    finally:
        for worker in WORKERS:
            if worker.device.type == 'cuda':
                with torch.cuda.device(worker.device):
                    torch.cuda.empty_cache()

if failures:
    atomic_write(OUTPUT_ROOT / '_failed.json', failures)

remaining = [v for v in video_dirs if not is_done(v.name)]
print(f'\nXong sau {(time.time() - started_all) / 60:.1f} phút: '
      f'success={success}, failed={failed}')
print(f'Tổng: {len(video_dirs) - len(remaining)}/{len(video_dirs)} video đã có caption.')
if remaining:
    print(f'CÒN {len(remaining)} video. Mở session mới chạy tiếp — nhớ Save Version rồi '
          'Add Input -> Your Work -> Notebook Output để không caption lại từ đầu.')
else:
    print('XONG TOÀN BỘ.')
pack_outputs()